In [ ]:
from ultralytics import YOLO
import cv2
from deep_sort_realtime.deepsort_tracker import DeepSort
import json
import uuid

# 🔹 Models
yolo_model = YOLO("/content/Cow_Detect/best.pt")
pose_model = YOLO("/content/Pose_Detect/best.pt")

KEYPOINT_ORDER = [ "right-ear", "nose", "left-ear", "neck", "left-front-hoof", "right-front-hoof", "hip", "left-back-hoof", "right-back-hoof", "tail" ]

tracker = DeepSort(max_age=30, n_init=3)

cap = cv2.VideoCapture("eat1.mp4")

frame_id = 0
final_output = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    # ---------------- YOLO DETECTION ----------------
    results = yolo_model(frame)[0]

    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        conf = float(box.conf[0])

        if conf < 0.5:
            continue

        detections.append((
            [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],   # width  = x2 - x1, height = y2 - y1
            conf,
            'cow'
        ))

    # ---------------- TRACKING ----------------
    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = map(int, track.to_ltrb())

        # ---------------- CROP ----------------
        cow_crop = frame[t:b, l:r]

        if cow_crop.shape[0] <= 0 or cow_crop.shape[1] <= 0:
            continue

        # ---------------- POSE ----------------
        pose_result = pose_model(cow_crop)[0]

        # Check if keypoints were detected and if there's at least one instance
        if pose_result.keypoints is None or len(pose_result.keypoints.xy) == 0:
            continue

        kps = pose_result.keypoints.xy[0].cpu().numpy()
        confs = pose_result.keypoints.conf[0].cpu().numpy()

        # ---------------- FORMAT OUTPUT ----------------
        keypoints = []
        for i, (kp, c) in enumerate(zip(kps, confs)):
            keypoints.append({
                "x": float(kp[0] + l),   # 🔹 convert to original frame
                "y": float(kp[1] + t),
                "confidence": float(c),
                "class_id": i,
                "class_name": KEYPOINT_ORDER[i]

            })

        final_output.append({
            "frame_id": frame_id,
            "track_id": track_id,
            "detection_id": str(uuid.uuid4()),
            "bbox": [float(x1), float(y1), float(x2), float(y2)],
            "keypoints": keypoints
        })

cap.release()

# ---------------- SAVE JSON ----------------
with open("output.json", "w") as f:
    json.dump({"predictions": final_output}, f, indent=2)


0: 384x640 1 COW, 146.5ms
Speed: 5.3ms preprocess, 146.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 COW, 148.5ms
Speed: 4.9ms preprocess, 148.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 COW, 222.6ms
Speed: 4.6ms preprocess, 222.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 544x640 1 cow, 323.2ms
Speed: 8.5ms preprocess, 323.2ms inference, 1.6ms postprocess per image at shape (1, 3, 544, 640)

0: 384x640 1 COW, 236.2ms
Speed: 7.8ms preprocess, 236.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 544x640 1 cow, 326.8ms
Speed: 5.8ms preprocess, 326.8ms inference, 2.5ms postprocess per image at shape (1, 3, 544, 640)

0: 384x640 1 COW, 208.1ms
Speed: 8.0ms preprocess, 208.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 544x640 1 cow, 302.3ms
Speed: 8.0ms preprocess, 302.3ms inference, 1.5ms postprocess per image at shape (1, 3, 544, 640)
